In [ ]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/raw/phdb_dinosaur_occurrences.csv')
df.head()

C:\Users\john_\AppData\Local\Temp\ipykernel_30488\3316100536.py:1: DtypeWarning: Columns (56,81,82,83,84,119,131) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/phdb_dinosaur_occurrences.csv')


,occurrence_no,record_type,collection_no,identified_name,identified_rank,accepted_name,accepted_attr,accepted_rank,accepted_no,early_interval,...,max_ma_error,rare_body_parts,min_ma_error,direct_ma_value,direct_ma_unit,direct_ma_method,zone_type,direct_ma_error,plant_organ,artifacts
0,41524,occ,3257,Aves indet.,class,Aves,(Linnaeus 1758),class,36616,Lutetian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41580,occ,3256,Aves indet.,class,Aves,(Linnaeus 1758),class,36616,Ypresian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,130209,occ,10755,Chaoyangosaurus liaosiensis n. gen. n. sp.,species,Chaoyangsaurus youngi,Zhao et al. 1999,species,55580,Late Kimmeridgian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,130294,occ,10764,Protarchaeopteryx robusta n. gen. n. sp.,species,Protarchaeopteryx robusta,Ji and Ji 1997,species,66068,Late Barremian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130295,occ,10764,Caudipteryx zoui n. gen. n. sp.,species,Caudipteryx zoui,Ji et al. 1998,species,66066,Late Barremian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The raw PBDB dataset contains many specialized fields that are not required
for the initial analysis.

Rather than removing fields solely based on missingness, fields will be
retained or removed based on their relevance to the project's analytical
questions.

The initial analytical domains are:

- Occurrence identification
- Taxonomy
- Geological time
- Geography
- Geology
- Paleoenvironment
- Preservation
- Data provenance

In [3]:
analysis_columns = [
    # Identification
    "occurrence_no",
    "record_type",
    "collection_no",
    "identified_name",
    "identified_rank",
    "accepted_name",
    "accepted_rank",
    "accepted_no",

    # Geological time
    "early_interval",
    "late_interval",
    "max_ma",
    "min_ma",

    # Taxonomy
    "phylum",
    "class",
    "order",
    "family",
    "genus",

    # Geography
    "cc",
    "state",
    "county",
    "lat",
    "lng",
    "latlng_basis",
    "latlng_precision",
    "geogscale",
    "paleolat",
    "paleolng",
    "geoplate",

    # Geology
    "formation",
    "geological_group",
    "stratscale",
    "lithology1",
    "lithology2",
    "environment",
    "tectonic_setting",

    # Preservation
    "pres_mode",
    "preservation_quality",
    "lagerstatten",
    "collection_coverage",
    "collection_type",

    # Provenance
    "ref_author",
    "ref_pubyr",
    "reference_no",
    "research_group"
]

In [4]:
missing_columns = [
    col for col in analysis_columns
    if col not in df.columns
]

missing_columns

[]

In [5]:
clean_df = df[analysis_columns].copy()

In [6]:
clean_df.shape

(37817, 44)

In [7]:
#Check for N/A values in the analysis columns
clean_df.isna().sum().sort_values(ascending=False).head(20)

tectonic_setting        36659
lagerstatten            36321
collection_coverage     35334
lithology2              31435
late_interval           29133
geological_group        27455
preservation_quality    26680
county                  19075
formation               14328
genus                   14244
paleolat                11704
paleolng                11704
family                   8733
stratscale               8381
order                    7385
geogscale                6629
state                    6319
research_group           1828
latlng_basis             1613
lithology1               1177
dtype: int64

In [8]:
#Check for blanks in analysis columns
(clean_df == "").sum().sort_values(ascending=False).head(20)

occurrence_no      0
record_type        0
collection_no      0
identified_name    0
identified_rank    0
accepted_name      0
accepted_rank      0
accepted_no        0
early_interval     0
late_interval      0
max_ma             0
min_ma             0
phylum             0
class              0
order              0
family             0
genus              0
cc                 0
state              0
county             0
dtype: int64

In [9]:
#Check for known variable 'not reported' in analysis columns
(clean_df == "not reported").sum().sort_values(ascending=False).head(20)

lithology1         12032
occurrence_no          0
collection_no          0
record_type            0
identified_rank        0
accepted_name          0
accepted_rank          0
accepted_no            0
early_interval         0
late_interval          0
max_ma                 0
identified_name        0
min_ma                 0
phylum                 0
order                  0
class                  0
genus                  0
cc                     0
state                  0
family                 0
dtype: int64

In [10]:
#Check lithology1 for values and fix not reported to NaN

clean_df = clean_df.replace("not reported", np.nan)
clean_df["lithology1"].value_counts(dropna=False).head(20)

lithology1
NaN                                13209
sandstone                          11079
"siliciclastic"                     2569
mudstone                            2092
claystone                           1876
siltstone                           1775
"limestone"                          937
"shale"                              611
conglomerate                         605
marl                                 593
tar                                  592
lime mudstone                        270
breccia                              241
"carbonate"                          201
gravel                               168
wackestone                           144
peat                                 124
"mixed carbonate-siliciclastic"      110
phosphorite                          108
grainstone                            93
Name: count, dtype: int64

In [11]:
#Check data types to prep for standarization
clean_df.dtypes

occurrence_no             int64
record_type              object
collection_no             int64
identified_name          object
identified_rank          object
accepted_name            object
accepted_rank            object
accepted_no               int64
early_interval           object
late_interval            object
max_ma                  float64
min_ma                  float64
phylum                   object
class                    object
order                    object
family                   object
genus                    object
cc                       object
state                    object
county                   object
lat                     float64
lng                     float64
latlng_basis             object
latlng_precision         object
geogscale                object
paleolat                float64
paleolng                float64
geoplate                 object
formation                object
geological_group         object
stratscale               object
litholog

In [12]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37817 entries, 0 to 37816
Data columns (total 44 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   occurrence_no         37817 non-null  int64  
 1   record_type           37817 non-null  object 
 2   collection_no         37817 non-null  int64  
 3   identified_name       37817 non-null  object 
 4   identified_rank       37817 non-null  object 
 5   accepted_name         37817 non-null  object 
 6   accepted_rank         37817 non-null  object 
 7   accepted_no           37817 non-null  int64  
 8   early_interval        37817 non-null  object 
 9   late_interval         8684 non-null   object 
 10  max_ma                37817 non-null  float64
 11  min_ma                37817 non-null  float64
 12  phylum                37817 non-null  object 
 13  class                 37817 non-null  object 
 14  order                 30432 non-null  object 
 15  family             

In [13]:
#Change id columns to string objects to not allow for any accidental math
id_columns = [
    "occurrence_no",
    "collection_no",
    "accepted_no",
    "reference_no"
]

clean_df[id_columns] = clean_df[id_columns].astype('string')
clean_df[id_columns].dtypes

occurrence_no    string[python]
collection_no    string[python]
accepted_no      string[python]
reference_no     string[python]
dtype: object

In [14]:
#Check expected numerical columns for typing
numeric_columns = [
    "max_ma",
    "min_ma",
    "lat",
    "lng",
    "paleolat",
    "paleolng",
    "ref_pubyr"
]

clean_df[numeric_columns].dtypes

max_ma       float64
min_ma       float64
lat          float64
lng          float64
paleolat     float64
paleolng     float64
ref_pubyr    float64
dtype: object

In [15]:
clean_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
max_ma,37817.0,71.029718,65.813394,0.011700,0.1290,72.200000,121.400000,251.902000
min_ma,37817.0,64.545258,61.613984,0.000000,0.0117,66.000000,100.500000,248.600000
lat,37817.0,24.991546,29.819721,-84.333336,18.3300,37.404999,43.575001,89.039169
lng,37817.0,-19.977060,92.787027,-179.154999,-106.6241,-17.209999,35.373600,178.677002
paleolat,26113.0,24.747404,32.048825,-86.160000,21.7200,33.890000,44.710000,89.200000
paleolng,26113.0,-0.107174,68.826948,-177.600000,-64.6100,3.070000,30.670000,178.700000
ref_pubyr,37816.0,1993.530093,29.454594,1824.000000,1988.0000,2001.000000,2012.000000,2027.000000


In [16]:
clean_df['ref_pubyr'] = clean_df['ref_pubyr'].astype('Int64')
clean_df["ref_pubyr"].dtype

Int64Dtype()

Data Quality Note 
'ref_pubyr' contains at least one publication year beyond the current calander year (2027). This value will be invested during validation rather than being removed automatically. 

In [17]:
#Checking Categorical values
categorical_columns = clean_df.select_dtypes(include="object").columns.tolist()

len(categorical_columns)

33

In [18]:
categorical_columns

['record_type',
 'identified_name',
 'identified_rank',
 'accepted_name',
 'accepted_rank',
 'early_interval',
 'late_interval',
 'phylum',
 'class',
 'order',
 'family',
 'genus',
 'cc',
 'state',
 'county',
 'latlng_basis',
 'latlng_precision',
 'geogscale',
 'geoplate',
 'formation',
 'geological_group',
 'stratscale',
 'lithology1',
 'lithology2',
 'environment',
 'tectonic_setting',
 'pres_mode',
 'preservation_quality',
 'lagerstatten',
 'collection_coverage',
 'collection_type',
 'ref_author',
 'research_group']

In [19]:
#Check and clean white space issues
whitespace_issues = {}

for col in categorical_columns:
    mask = clean_df[col].astype("string").str.strip() != clean_df[col].astype("string")
    count = mask.sum()

    if count > 0:
        whitespace_issues[col] = count

whitespace_issues

{'state': np.int64(11),
 'county': np.int64(4),
 'formation': np.int64(5),
 'geological_group': np.int64(3),
 'ref_author': np.int64(9)}

In [20]:
for col in whitespace_issues:
    print(f"\n--- {col} ---")
    
    mask = (
        clean_df[col].astype("string").str.strip()
        != clean_df[col].astype("string")
    )
    
    print(clean_df.loc[mask, col].unique())


--- state ---
[" Provence-Alpes-Côte d'Azur" 'Tarija ' ' Gyeongsangnam-do'
 'Gyeongsangnam-do ' 'central Macedonia ' 'Nouvelle-Aquitaine ' ' Neuquén'
 'Guizhou ' 'Jiangxi ' 'Chongqing ']

--- county ---
['Rockingham ' 'Baranya ' 'Huichang ' 'Middlesex ']

--- formation ---
['Mesilla Valley ' 'Itanoura ' 'Menilite ' 'Hudspeth ']

--- geological_group ---
['El Foyel ' 'Carmanah ']

--- ref_author ---
['Desjardins ']


In [21]:
clean_df[categorical_columns] = clean_df[categorical_columns].apply(
    lambda col: col.str.strip()
)

fixed_whitespace_issues = {}

for col in categorical_columns:
    mask = (
        clean_df[col].astype("string").str.strip()
        != clean_df[col].astype("string")
    )
    
    count = mask.sum()
    
    if count > 0:
        whitespace_issues[col] = count

fixed_whitespace_issues

{}

In [22]:
category_counts = (
    clean_df[categorical_columns]
    .nunique(dropna=True)
    .sort_values()
)

category_counts

record_type                 1
phylum                      1
lagerstatten                2
class                       4
geogscale                   5
stratscale                  5
latlng_basis                5
preservation_quality        6
collection_type             6
tectonic_setting           11
latlng_precision           11
accepted_rank              15
identified_rank            15
research_group             27
collection_coverage        28
lithology2                 30
lithology1                 41
order                      68
environment                70
geoplate                   72
late_interval             154
pres_mode                 160
cc                        168
early_interval            253
geological_group          314
family                    409
state                     876
county                   1305
formation                1776
genus                    3087
ref_author               3818
accepted_name            6172
identified_name         10505
dtype: int

In [23]:
# Manually check smaller category counts for any erronous values
for col in category_counts[category_counts <= 25].index:
    print(f"\n--- {col} ---")
    print(clean_df[col].value_counts(dropna=False))


--- record_type ---
record_type
occ    37817
Name: count, dtype: int64

--- phylum ---
phylum
Chordata    37817
Name: count, dtype: int64

--- lagerstatten ---
lagerstatten
NaN             36321
concentrate      1349
conservation      147
Name: count, dtype: int64

--- class ---
class
Aves            15176
Reptilia        11176
Ornithischia     7063
Saurischia       4402
Name: count, dtype: int64

--- geogscale ---
geogscale
small collection    14516
outcrop             14406
NaN                  6629
local area           2074
hand sample            98
basin                  94
Name: count, dtype: int64

--- stratscale ---
stratscale
bed              20407
NaN               8381
group of beds     7142
formation         1115
member             708
group               64
Name: count, dtype: int64

--- latlng_basis ---
latlng_basis
based on nearby landmark    12868
estimated from map          11994
stated in text               9005
NaN                          1613
based on political uni

##### No capitalization inconsistences found
##### No obvious spelling inconsistences found
##### Scientific categories preserved. 

##### latlng_precision needs further investigation to determine full meaning.
##### lagerstatten may be converted to a True/False category for ease.

In [25]:
clean_df.duplicated().sum()

np.int64(0)

In [26]:
clean_df["occurrence_no"].duplicated().sum()

np.int64(0)

In [27]:
for col in ["occurrence_no", "collection_no", "accepted_no", "reference_no"]:
    print(f"{col}: {clean_df[col].nunique()} unique values")

occurrence_no: 37817 unique values
collection_no: 14351 unique values
accepted_no: 6175 unique values
reference_no: 7781 unique values
